# Pruebas Manuales v2
Cubre 5 etapas del pipeline en orden:
1. Dataset crudo (CSV)
2. Preprocesamiento (NaN, rangos)
3. Ventanas y split
4. Módulos `src/` (shapes y contratos)
5. Resumen final

In [ ]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

RESULTADOS = []  # acumula (nombre, ok, detalle)

def check(nombre, condicion, detalle=''):
    estado = '✓' if condicion else '✗'
    RESULTADOS.append((nombre, condicion, detalle))
    print(f'  [{estado}] {nombre}' + (f' — {detalle}' if detalle else ''))
    if not condicion:
        raise AssertionError(f'FALLÓ: {nombre}. {detalle}')

print('Setup OK')

## 1 — Dataset crudo

In [ ]:
print('=== 1. DATASET CRUDO ===')

CSV_PATH = 'industrial_dataset.csv'
COLUMNAS_ESPERADAS = 'vibration,current,speed,temperature,micro_stop,cause'

check('CSV existe', os.path.exists(CSV_PATH))

with open(CSV_PATH) as f:
    encabezado = f.readline().strip()
check('Encabezado correcto', encabezado == COLUMNAS_ESPERADAS, encabezado)

dataset = np.genfromtxt(CSV_PATH, delimiter=',', skip_header=1)
check('Shape (20000, 6)', dataset.shape == (20000, 6), str(dataset.shape))

total_micro = int(dataset[:, 4].sum())
check('Micro-paradas > 0', total_micro > 0, f'{total_micro} eventos')
check('Micro-paradas < 5% del total', total_micro < 20000 * 0.05,
      f'{total_micro/200:.2f}%')

causas = np.unique(dataset[:, 5])
check('Causas válidas [0-4]', np.all(np.isin(causas, [0,1,2,3,4])), str(causas))

nan_crudo = int(np.isnan(dataset[:, :4]).sum())
check('NaN sólo en columnas de sensor (cols 0-3)',
      np.isnan(dataset[:, 4:]).sum() == 0,
      f'{nan_crudo} NaN en sensores, 0 en etiquetas')
check('NaN dentro del rango esperado (0.1% – 5%)',
      0.001 * 20000 * 4 <= nan_crudo <= 0.05 * 20000 * 4,
      f'{nan_crudo} NaN = {nan_crudo/(20000*4)*100:.2f}%')

## 2 — Preprocesamiento

In [ ]:
print('=== 2. PREPROCESAMIENTO ===')

# Interpolación lineal (replicando lo que hace dataset.ipynb)
data = dataset.copy()
for col in range(4):
    serie = data[:, col]
    idx = np.arange(len(serie))
    validos = ~np.isnan(serie)
    data[:, col] = np.interp(idx, idx[validos], serie[validos])

check('Sin NaN tras interpolación', np.isnan(data[:, :4]).sum() == 0)

# Rangos físicos plausibles (columnas: vibration, current, speed, temperature)
RANGOS = {
    'vibration':   (-20,  20),
    'current':     (  0,  20),
    'speed':       (-50, 200),
    'temperature': (  0, 200),
}
nombres_col = list(RANGOS.keys())
for i, (nombre, (lo, hi)) in enumerate(RANGOS.items()):
    col = data[:, i]
    check(f'{nombre} en [{lo}, {hi}]',
          float(col.min()) >= lo and float(col.max()) <= hi,
          f'min={col.min():.2f}, max={col.max():.2f}')

# Normalización min-max → [0, 1]
data_norm = data.copy()
for i in range(4):
    lo, hi = data[:, i].min(), data[:, i].max()
    data_norm[:, i] = (data[:, i] - lo) / (hi - lo)

check('Normalización produce valores en [0, 1]',
      data_norm[:, :4].min() >= 0 and data_norm[:, :4].max() <= 1)

## 3 — Ventanas y split

In [ ]:
print('=== 3. VENTANAS Y SPLIT ===')

WINDOW = 20
features = data_norm[:, :4].astype(np.float32)
labels   = data_norm[:, 4].astype(int)

X_wins, y_wins = [], []
for i in range(len(features) - WINDOW):
    X_wins.append(features[i : i + WINDOW])
    y_wins.append(labels[i + WINDOW])
X_wins = np.array(X_wins)
y_wins = np.array(y_wins)

n_esperado = 20000 - WINDOW
check(f'Número de ventanas = {n_esperado}',
      len(X_wins) == n_esperado, str(len(X_wins)))
check('Shape ventana (N, 20, 4)',
      X_wins.shape == (n_esperado, WINDOW, 4), str(X_wins.shape))
check('Sin NaN en ventanas', np.isnan(X_wins).sum() == 0)
check('Etiquetas binarias (solo 0 y 1)',
      set(np.unique(y_wins)).issubset({0, 1}))

# Split 70/15/15
N = len(X_wins)
train_end = int(N * 0.70)
val_end   = int(N * 0.85)
X_train, X_val, X_test = X_wins[:train_end], X_wins[train_end:val_end], X_wins[val_end:]
y_train, y_val, y_test = y_wins[:train_end], y_wins[train_end:val_end], y_wins[val_end:]

check('Split suma el total', len(X_train)+len(X_val)+len(X_test) == N,
      f'{len(X_train)}+{len(X_val)}+{len(X_test)}={len(X_train)+len(X_val)+len(X_test)}')
check('Train ≈ 70%', 0.68 <= len(X_train)/N <= 0.72,
      f'{len(X_train)/N*100:.1f}%')
check('Val ≈ 15%',   0.13 <= len(X_val)/N   <= 0.17,
      f'{len(X_val)/N*100:.1f}%')
check('Test ≈ 15%',  0.13 <= len(X_test)/N  <= 0.17,
      f'{len(X_test)/N*100:.1f}%')
check('Micro-paradas en train > 0', y_train.sum() > 0, str(int(y_train.sum())))
check('Micro-paradas en test > 0',  y_test.sum()  > 0, str(int(y_test.sum())))

## 4 — Módulos src/

In [ ]:
print('=== 4. MÓDULOS SRC/ ===')

from neural_engine.lstm_cell import LSTMCell, AdamOptimizer
from neural_engine.attention import Attention
from neural_engine.prediction_head import PredictionHead

check('Importación de módulos OK', True)

np.random.seed(0)
X_prueba = np.random.randn(8, 20, 4)

lstm = LSTMCell(input_size=4, hidden_size=32)
attn = Attention(hidden_size=32)
head = PredictionHead(hidden_size=32)

all_h, h_last = lstm.forward(X_prueba)
check('LSTMCell.forward → all_h (8, 20, 32)', all_h.shape == (8, 20, 32), str(all_h.shape))
check('LSTMCell.forward → h_last (8, 32)',    h_last.shape == (8, 32),    str(h_last.shape))
check('Sin NaN en all_h', not np.any(np.isnan(all_h)))

context, alpha = attn.forward(all_h)
check('Attention.forward → context (8, 32)', context.shape == (8, 32), str(context.shape))
check('Attention.forward → alpha (8, 20)',   alpha.shape   == (8, 20), str(alpha.shape))
check('Alpha suma 1 por fila', np.allclose(alpha.sum(axis=1), 1.0, atol=1e-5))

pred = head.forward(context)
check('PredictionHead.forward → pred (8,)', pred.shape == (8,), str(pred.shape))
check('Pred ≥ 0 (ReLU activo)',             pred.min() >= 0,    f'min={pred.min():.4f}')

# Contrato extract_context
def extract_context(X, lstm, attn):
    all_h, _ = lstm.forward(X)
    ctx, _   = attn.forward(all_h)
    return ctx

ctx = extract_context(X_prueba, lstm, attn)
check('Contrato extract_context → (8, 32)', ctx.shape == (8, 32), str(ctx.shape))

In [ ]:
print('=== 4b. GRADIENT FLOW ===')

opt    = AdamOptimizer(lr=1e-2)
lstm2  = LSTMCell(input_size=4, hidden_size=32)
attn2  = Attention(hidden_size=32)
head2  = PredictionHead(hidden_size=32)
y_true = np.random.uniform(0, 10, size=8)
losses = []

for _ in range(20):
    all_h2, _  = lstm2.forward(X_prueba)
    ctx2, _    = attn2.forward(all_h2)
    pred2      = head2.forward(ctx2)
    losses.append(head2.loss(pred2, y_true))
    dL_dctx  = head2.backward(pred2, y_true)
    dL_dallh = attn2.backward(dL_dctx)
    grads    = lstm2.backward(dL_dallh)
    lstm2.update(grads, opt)

check('Loss decrece en 20 iteraciones', losses[-1] < losses[0],
      f'{losses[0]:.4f} → {losses[-1]:.4f}')

## 5 — Resumen final

In [ ]:
print('=' * 50)
print('RESUMEN DE PRUEBAS MANUALES v2')
print('=' * 50)

aprobadas = sum(1 for _, ok, _ in RESULTADOS if ok)
total     = len(RESULTADOS)

for nombre, ok, detalle in RESULTADOS:
    estado = '✓' if ok else '✗'
    print(f'  [{estado}] {nombre}' + (f' — {detalle}' if detalle else ''))

print()
print(f'Resultado: {aprobadas}/{total} pruebas aprobadas')

if aprobadas == total:
    print('Todo OK ✓')
else:
    fallidas = [n for n, ok, _ in RESULTADOS if not ok]
    print('Fallidas:', fallidas)